# Score the reference model

Loads the 2,000 saved samples from `01` and computes FID and KID against all 4,317 real images, plus the real-vs-real floor.

In [1]:
!pip install -q torch-fidelity

In [2]:
import sys, glob, json, numpy as np, torch, torchvision, kagglehub
import torchvision.transforms as T
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

DEV = torch.device("cuda")
f_npy = glob.glob("/kaggle/input/**/baseline_asis/samples_seed0.npy", recursive=True)
print(f_npy)
gen = torch.from_numpy(np.load(f_npy[0]))
print("generated:", gen.shape, gen.dtype)

path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
tf = T.Compose([T.ToTensor(), T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)])
ds = torchvision.datasets.ImageFolder(root=path, transform=tf)
real_u8 = (torch.stack([ds[i][0] for i in range(len(ds))]).clamp(0, 1) * 255).round().to(torch.uint8)
print("real:", real_u8.shape)

def fid_kid(a_u8, b_u8):
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 500):
            xb = x[i:i+500].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute()
    return fid.compute().item(), km.item(), ks.item()

f, km, ks = fid_kid(real_u8, gen)
perm = torch.randperm(len(real_u8), generator=torch.Generator().manual_seed(0)); h = len(real_u8) // 2
ff, fk, _ = fid_kid(real_u8[perm[:h]], real_u8[perm[h:2*h]])
res = {"run": "A_baseline_published_config", "n_gen": len(gen), "seed": 0,
       "fid": f, "kid": km, "kid_std": ks, "floor_fid": ff, "floor_kid": fk}
json.dump(res, open("/kaggle/working/baseline_results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

['/kaggle/input/notebooks/malharinamdar/notebook11174aea07/baseline_asis/samples_seed0.npy']
generated: torch.Size([2000, 3, 32, 32]) torch.uint8
real: torch.Size([4317, 3, 32, 32])


{
  "run": "A_baseline_published_config",
  "n_gen": 2000,
  "seed": 0,
  "fid": 248.5438232421875,
  "kid": 0.16783946752548218,
  "kid_std": 0.0031570708379149437,
  "floor_fid": 22.614994049072266,
  "floor_kid": -2.0563602447509766e-05
}
